# chemistree benchmark — statistics & reproducibility

Recomputes every statistic in the writeup **from the raw per-case result rows** in
`benchmarks/results/full/`. The pairing/collapse logic is written out by hand so it is
checkable; the exact test itself uses `scipy.stats.binomtest`. `pandas` is used only to
render tables. (If `pandas`/`scipy` are missing, `pip install pandas scipy` — they are not
project dependencies.)

Run top-to-bottom, from the repo root or from `notebooks/`.

## Method

- **Data.** One row per *(arm, track, target, case, replicate)*, read from
  `results/full/<target>/<track>_<arm>.jsonl`. Model = haiku, **N = 3 replicates per case**.
- **Tracks.** `2d` (editing, binary `correct`), `probes` (3D Q&A, binary `correct`),
  `decorate` (3D decoration).
- **Success ("solved").** `correct` for `2d`/`probes`; for `decorate`, **improved predicted
  binding vs the scaffold** (`redock_delta < 0`). A run that never reached docking counts as a
  failure. Whether a run merely *reached docking* is reported separately as **completion**
  (`status == "ok"` and a finite `redock_final`) — a useful statistic, but not the success
  criterion.
- **Pairing.** Every arm runs the identical case, so arm-vs-arm is compared **per case**. The
  **three replicates are collapsed** to a per-arm success-count (0–3) per case — never counted as
  three independent observations (that would pseudo-replicate and overstate significance).
- **Test.** Exact two-sided **sign test** on the discordant cases (a case is a win for the arm
  that succeeded on more of its three replicates; ties dropped). The p-value is the exact
  two-sided binomial probability, computed with **`scipy.stats.binomtest(wins, discordant, 0.5)`**.
- **Tokens.** `input_tokens` / `output_tokens` are the raw usage fields; `context_footprint`
  (uncached + cache-create + cache-read) is the *total* context processed — reported alongside
  because chemistree's input is mostly cache reads, so `input_tokens` alone understates it.
- **naked has no 3D** (2D only); any comparison involving naked uses 2D cases only.

In [7]:
import json, collections
from pathlib import Path
from statistics import mean
import pandas as pd
from scipy.stats import binomtest

# Locate benchmarks/results/full whether run from the repo root or from notebooks/
root = Path.cwd()
for _ in range(4):
    if (root / "benchmarks" / "results" / "full").is_dir():
        break
    root = root.parent
RESULTS = root / "benchmarks" / "results" / "full"
assert RESULTS.is_dir(), f"results/full not found from {Path.cwd()}"

def load_rows():
    # One dict per result row, tagged with target / track / arm from its file path.
    # The ablation is flagged by each row's own boolean 'guidance' field (True = guided).
    rows = []
    for f in sorted(RESULTS.glob("*/*.jsonl")):
        target, name = f.parent.name, f.stem
        if   name.startswith("2d_"):       track, arm = "2d",       name[3:]
        elif name.startswith("probes_"):   track, arm = "probes",   name[7:]
        elif name.startswith("decorate_"): track, arm = "decorate", name[9:]
        else:                              continue
        arm = arm.replace("_noguid", "")
        for line in f.read_text().splitlines():
            if line.strip():
                r = json.loads(line)
                r["target"], r["track"], r["arm"] = target, track, arm
                rows.append(r)
    return rows

ROWS = load_rows()
print(len(ROWS), "rows |", {t: sum(r["track"] == t for r in ROWS) for t in ("2d", "probes", "decorate")})
print("arms:", sorted({r["arm"] for r in ROWS}),
      "| targets:", len({r["target"] for r in ROWS}),
      "| replicates:", sorted({r["replicate"] for r in ROWS}))

495 rows | {'2d': 243, 'probes': 120, 'decorate': 132}
arms: ['chemistree', 'generalist', 'naked'] | targets: 11 | replicates: [1, 2, 3]


In [ ]:
# ---------- how success and the paired statistics are computed ----------

def solved(r):
    # SUCCESS. 2d/probes: the binary 'correct'.
    # decorate: improved predicted binding vs the scaffold (redock_delta < 0);
    #           a run that never docked (redock_delta is None) is a failure.
    if r["track"] == "decorate":
        return int(r.get("redock_delta") is not None and r["redock_delta"] < 0)
    return int(bool(r.get("correct")))

def completed(r):
    # Reached docking at all (a scoreable final molecule) -- reported, not the success metric.
    return int(r.get("status") == "ok" and r.get("redock_final") is not None)

def avg(rows, key, nd=None):
    # Mean over rows that actually have the field (skips incomplete/missing runs).
    xs = [r[key] for r in rows if r.get(key) is not None]
    if not xs:
        return None
    m = mean(xs)
    return round(m, nd) if nd is not None else round(m)

def sign_p(wins_a, wins_b):
    # Exact two-sided sign test = exact two-sided binomial test on the discordant pairs.
    n = wins_a + wins_b
    if n == 0:
        return 1.0
    return float(binomtest(wins_a, n, 0.5, alternative="two-sided").pvalue)

def case_counts(arm, tracks, guided=True):
    # Collapse the replicates: per (track, target, case) -> # of the arm's successes (0..N).
    c = collections.Counter()
    for r in ROWS:
        if r["arm"] == arm and r["track"] in tracks and bool(r.get("guidance")) == guided:
            c[(r["track"], r["target"], r["id"])] += solved(r)
    return c

def paired_sign_test(arm_a, arm_b, tracks, guided=True):
    # Pair by case (both arms ran it); a case is a win for whichever arm succeeded on more of
    # its replicates; ties (equal counts) are concordant and dropped.
    A, B = case_counts(arm_a, tracks, guided), case_counts(arm_b, tracks, guided)
    keys = set(A) & set(B)
    wa = sum(A[k] > B[k] for k in keys)
    wb = sum(B[k] > A[k] for k in keys)
    return {"a": arm_a, "b": arm_b, "wins_a": wa, "wins_b": wb,
            "discordant": wa + wb, "paired_cases": len(keys),
            "p_exact": round(sign_p(wa, wb), 4)}

def agg(arm, tracks, guided=True):
    # Success/total, success rate, and mean tokens/cost over an arm's rows in the given tracks.
    rs = [r for r in ROWS if r["arm"] == arm and r["track"] in tracks
          and bool(r.get("guidance")) == guided]
    if not rs:
        return None
    s = sum(solved(r) for r in rs)
    return {"arm": arm, "success": s, "n": len(rs), "success_rate": round(s / len(rs), 3),
            "mean_input_tok": avg(rs, "input_tokens"),
            "mean_output_tok": avg(rs, "output_tokens"),
            "mean_footprint": avg(rs, "context_footprint"),
            "mean_cost_usd": avg(rs, "cost_usd", 4)}

COLS = ["arm", "success", "success_rate", "mean_input_tok", "mean_output_tok", "mean_footprint", "mean_cost_usd"]
def table(arms, tracks):
    t = pd.DataFrame([agg(a, tracks) for a in arms])
    t["success"] = t["success"].astype(str) + "/" + t["n"].astype(str)
    return t[COLS]

## Overall (all three tracks: 2D editing + 3D comprehension + 3D decoration)

The full benchmark set, success scored per track (correct SMILES / matching answer / improved
binding `Δbinding < 0`; decoration is the **guided** condition). `naked` has no 3D access, so its
row is 2D-only (n = 81); the tool-using arms run all three tracks (n = 174). Per-track
breakdowns follow. Overall token/cost include the expensive decoration track, so they exceed any
single track's.

In [9]:
display(table(["naked", "generalist", "chemistree"], ("2d", "probes", "decorate")))
print("naked vs chemistree      (2D only, naked has no 3D)     :", paired_sign_test("naked", "chemistree", ("2d",)))
print("generalist vs chemistree (all three tracks)             :", paired_sign_test("generalist", "chemistree", ("2d", "probes", "decorate")))
print("generalist vs chemistree (2D + probes only, sensitivity):", paired_sign_test("generalist", "chemistree", ("2d", "probes")))

,arm,success,success_rate,mean_input_tok,mean_output_tok,mean_footprint,mean_cost_usd
0,naked,55/81,0.679,11,10840,16069,0.0628
1,generalist,122/174,0.701,60,8433,206876,0.0897
2,chemistree,153/174,0.879,129,4875,495397,0.1033


naked vs chemistree      (2D only, naked has no 3D)     : {'a': 'naked', 'b': 'chemistree', 'wins_a': 1, 'wins_b': 10, 'discordant': 11, 'paired_cases': 27, 'p_exact': 0.0117}
generalist vs chemistree (all three tracks)             : {'a': 'generalist', 'b': 'chemistree', 'wins_a': 4, 'wins_b': 24, 'discordant': 28, 'paired_cases': 58, 'p_exact': 0.0002}
generalist vs chemistree (2D + probes only, sensitivity): {'a': 'generalist', 'b': 'chemistree', 'wins_a': 3, 'wins_b': 18, 'discordant': 21, 'paired_cases': 47, 'p_exact': 0.0015}


## 2D editing only

Full 2D suite (27 cases x 3 = 81 rows), the category breakdown, and all three pairwise
sign tests.

In [10]:
display(table(["naked", "generalist", "chemistree"], ("2d",)))

cats = ["substitution", "growing", "core_hopping"]
catrows = []
for a in ("naked", "generalist", "chemistree"):
    rd = {"arm": a}
    for c in cats:
        rs = [r for r in ROWS if r["arm"] == a and r["track"] == "2d" and r.get("category") == c]
        rd[c] = f"{sum(solved(r) for r in rs)}/{len(rs)}"
    catrows.append(rd)
print("2D accuracy by category (solved / total):")
display(pd.DataFrame(catrows))

print("sign tests (2D, case-level):")
for a, b in [("naked", "chemistree"), ("generalist", "chemistree"), ("generalist", "naked")]:
    print(" ", paired_sign_test(a, b, ("2d",)))

,arm,success,success_rate,mean_input_tok,mean_output_tok,mean_footprint,mean_cost_usd
0,naked,55/81,0.679,11,10840,16069,0.0628
1,generalist,52/81,0.642,33,7144,93841,0.0666
2,chemistree,72/81,0.889,67,4062,166189,0.0582


2D accuracy by category (solved / total):


,arm,substitution,growing,core_hopping
0,naked,24/30,17/24,14/27
1,generalist,25/30,15/24,12/27
2,chemistree,29/30,21/24,22/27


sign tests (2D, case-level):
  {'a': 'naked', 'b': 'chemistree', 'wins_a': 1, 'wins_b': 10, 'discordant': 11, 'paired_cases': 27, 'p_exact': 0.0117}
  {'a': 'generalist', 'b': 'chemistree', 'wins_a': 1, 'wins_b': 13, 'discordant': 14, 'paired_cases': 27, 'p_exact': 0.0018}
  {'a': 'generalist', 'b': 'naked', 'wins_a': 4, 'wins_b': 7, 'discordant': 11, 'paired_cases': 27, 'p_exact': 0.5488}


## 3D probes

Deterministic Q&A over a fixed pose (20 cases x 3 = 60 rows), chemistree vs generalist.

In [11]:
display(table(["generalist", "chemistree"], ("probes",)))
print("generalist vs chemistree (probes):", paired_sign_test("generalist", "chemistree", ("probes",)))

,arm,success,success_rate,mean_input_tok,mean_output_tok,mean_footprint,mean_cost_usd
0,generalist,55/60,0.917,40,3655,98697,0.0446
1,chemistree,57/60,0.950,51,1432,116073,0.0343


generalist vs chemistree (probes): {'a': 'generalist', 'b': 'chemistree', 'wins_a': 2, 'wins_b': 5, 'discordant': 7, 'paired_cases': 20, 'p_exact': 0.4531}


## 3D decoration

**Success = improved predicted binding** (redock `Δbinding < 0` vs the scaffold). **Completion**
(reached docking at all) is shown as a separate column — a useful statistic, not the success
criterion. Also shown: mean Δbinding, ECFP4 recovery, and the chemistree `write_pose` pose
score (over completed runs), the co-completed head-to-head, and the success sign test.

In [12]:
rows = []
for arm in ("chemistree", "generalist"):
    for guided in (True, False):
        rs = [r for r in ROWS if r["arm"] == arm and r["track"] == "decorate"
              and bool(r.get("guidance")) == guided]
        comp = [r for r in rs if completed(r)]
        rows.append({"arm": arm, "guidance": "guided" if guided else "ablation",
                     "success(d<0)": f"{sum(solved(r) for r in rs)}/{len(rs)}",
                     "completed": f"{len(comp)}/{len(rs)}",
                     "mean_dbind": avg(comp, "redock_delta", 3),
                     "mean_recovery": avg(comp, "recovery_final", 3),
                     "mean_pose": avg(comp, "pose_final", 3),
                     "mean_cost_usd": avg(rs, "cost_usd", 4)})
display(pd.DataFrame(rows))

print("SUCCESS (improved binding) sign test, chemistree vs generalist (guided):",
      paired_sign_test("chemistree", "generalist", ("decorate",), guided=True))

# Co-completed (guided): on (target, replicate) pairs where BOTH arms docked, who docks better?
cg = {(r["target"], r["replicate"]): r for r in ROWS
      if r["arm"] == "chemistree" and r["track"] == "decorate" and r.get("guidance")}
gg = {(r["target"], r["replicate"]): r for r in ROWS
      if r["arm"] == "generalist" and r["track"] == "decorate" and r.get("guidance")}
both = [k for k in cg if k in gg
        and cg[k].get("redock_final") is not None and gg[k].get("redock_final") is not None]
chem_better = sum((cg[k].get("redock_delta") or 0) < (gg[k].get("redock_delta") or 0) for k in both)
dd = round(mean((cg[k].get("redock_delta") or 0) - (gg[k].get("redock_delta") or 0) for k in both), 3)
print(f"co-completed guided pairs: {len(both)} | chemistree docks better on {chem_better}/{len(both)}"
      f" | mean delta-delta (chem - gen) = {dd}")

,arm,guidance,success(d<0),completed,mean_dbind,mean_recovery,mean_pose,mean_cost_usd
0,chemistree,guided,24/33,32/33,-0.656,0.311,-8.098,0.3395
1,chemistree,ablation,22/33,33/33,-0.444,0.359,-7.516,0.2022
2,generalist,guided,15/33,23/33,-0.430,0.288,NaN,0.2804
3,generalist,ablation,14/33,22/33,-0.791,0.267,NaN,0.1442


SUCCESS (improved binding) sign test, chemistree vs generalist (guided): {'a': 'chemistree', 'b': 'generalist', 'wins_a': 6, 'wins_b': 1, 'discordant': 7, 'paired_cases': 11, 'p_exact': 0.125}
co-completed guided pairs: 23 | chemistree docks better on 12/23 | mean delta-delta (chem - gen) = 0.013
